# R6 step 1: planner feasibility (GPU runtime)

Evaluates two public-layout checkpoints **with planning** through the public tdmpc2
code's own `TDMPC2` class (tdmpc2 e9f59321, `api_model_conversion` included):
cartpole-swingup seed 2 and humanoid-run seed 1, 5 episodes each, in both
`eval_mode=True` (as the training-time evaluations behind the published curves) and
`eval_mode=False` (as `evaluate.py`). Reports returns against the published ones and
seconds per episode.

No lens, coverage or criterion quantity is computed.

**Runtime > Change runtime type > GPU.** tdmpc2 runs in its own Python 3.11
virtualenv with the exact pins of its `docker/environment.yaml`; the project's JAX
environment is not used here.

In [ ]:
# 1. Repository with tags (GITHUB_TOKEN Colab secret if the repository is private).
import os, subprocess
BRANCH = "claude/loving-johnson-wk9yb3"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = (f"https://{token}@github.com/binoygeorge97/layernorm-lens" if token
       else "https://github.com/binoygeorge97/layernorm-lens")
if not os.path.isdir("/content/layernorm-lens"):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, "/content/layernorm-lens"], check=True)
%cd /content/layernorm-lens
!git fetch --tags -q origin
!git log --oneline -1
!for t in prereg-r6 prereg-r6-d1 prereg-r6-d2; do printf "%-14s " $t; git cat-file -t $t && git rev-parse $t^{commit}; done
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
# 2. tdmpc2 at the pinned commit.
!test -d checkpoints/tdmpc2_src || git clone -q https://github.com/nicklashansen/tdmpc2 checkpoints/tdmpc2_src
!git -C checkpoints/tdmpc2_src checkout -q e9f59321933cbc8e11a002b842adc7d4ffae8ff1 && git -C checkpoints/tdmpc2_src rev-parse HEAD

In [ ]:
# 3. Python 3.11 virtualenv with tdmpc2's pinned pip dependencies (docker/environment.yaml).
!pip install -q uv
!uv venv -q --python 3.11 /content/tdmpc2-venv
import yaml
env = yaml.safe_load(open("checkpoints/tdmpc2_src/docker/environment.yaml"))
pins = [p for d in env["dependencies"] if isinstance(d, dict) for p in d["pip"]]
print(pins)
open("/content/tdmpc2-pins.txt", "w").write("\n".join(pins + ["pyyaml"]) + "\n")
!uv pip install -q --python /content/tdmpc2-venv/bin/python -r /content/tdmpc2-pins.txt
!/content/tdmpc2-venv/bin/python -c "import torch, tensordict, torchrl, dm_control, mujoco, numpy; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'tensordict', tensordict.__version__, 'torchrl', torchrl.__version__, 'mujoco', mujoco.__version__, 'numpy', numpy.__version__)"

In [ ]:
# 4. The two checkpoints, from the pinned Hugging Face revision (same layout as extract.py).
import json, os, hashlib, urllib.request
cfg = yaml.safe_load(open("experiments/r6_tdmpc2/config.yaml"))
rev = cfg["source"]["hf_revision"]
repo = cfg["source"]["checkpoints"].split("huggingface.co/")[1]
files = [s["rfilename"] for s in json.load(urllib.request.urlopen(
    f"https://huggingface.co/api/models/{repo}/revision/{rev}"))["siblings"]]
for item in cfg["planner_check"]["checkpoints"]:
    name = f"{item['task']}-{item['seed']}.pt"
    hits = [f for f in files if os.path.basename(f) == name]
    assert len(hits) == 1, (name, hits)
    local = os.path.join(cfg["paths"]["checkpoints"], hits[0])
    if not os.path.exists(local):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        urllib.request.urlretrieve(f"https://huggingface.co/{repo}/resolve/{rev}/{hits[0]}", local)
    print(local, hashlib.sha256(open(local, "rb").read()).hexdigest())

In [ ]:
# 5. Planner evaluation through tdmpc2's TDMPC2 class. The first episode of each
#    checkpoint includes torch.compile time.
!cd /content/layernorm-lens && MUJOCO_GL=egl /content/tdmpc2-venv/bin/python experiments/r6_tdmpc2/planner_check.py --config experiments/r6_tdmpc2/config.yaml

In [ ]:
# 6. Results, and the files to commit (small).
import pandas as pd
display(pd.read_csv("results/r6/planner_check.csv"))
!zip -q r6_planner_check.zip results/r6/planner_check.csv results/r6/meta_planner_check.json
from google.colab import files
files.download("r6_planner_check.zip")